In [ ]:
# HY-ARIA Robust CNN Trainer (Transfer Learning)
Because Windows restricts path lengths and local CPU training of 87,000 image datasets can take up to 6 hours, it's highly recommended to run this notebook in **Google Colab** (which provides free GPU compute and Linux architecture).

**Instructions:**
1. Click `Runtime > Change runtime type` and select **T4 GPU**.
2. Click `Runtime > Run all`.
3. Wait ~15 minutes.
4. Download `robust_vision_model.keras` from the files pane on the left.
5. Place it in your local `hybrid/backend/models` directory.

In [ ]:
!pip install kagglehub

import os
import json
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import kagglehub

print("Libraries loaded. Fetching Massive Kaggle Dataset...")
# This circumvents Windows Path limits entirely because it's running on a Linux cloud container
dataset_path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
base_dir = os.path.join(dataset_path, "New Plant Diseases Dataset(Augmented)", "New Plant Diseases Dataset(Augmented)")

train_dir = os.path.join(base_dir, "train")
valid_dir = os.path.join(base_dir, "valid")

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 10

print("Loading optimized tensors...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir, shuffle=True, batch_size=BATCH_SIZE, image_size=IMG_SIZE
)

valid_dataset = tf.keras.utils.image_dataset_from_directory(
    valid_dir, shuffle=True, batch_size=BATCH_SIZE, image_size=IMG_SIZE
)

class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Loaded {num_classes} classes.")

with open('robust_disease_classes.json', 'w') as f:
    json.dump(class_names, f)

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
valid_dataset = valid_dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
print("Building Transfer Learning Model (MobileNetV2)...")

base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False 

inputs = tf.keras.Input(shape=(224, 224, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model_checkpoint = ModelCheckpoint('robust_vision_model.keras', save_best_only=True, monitor='val_accuracy')
early_stopping = EarlyStopping(patience=3, restore_best_weights=True)

print("Training robust classification head (Epochs 1-10)...")
model.fit(train_dataset, validation_data=valid_dataset, epochs=EPOCHS, callbacks=[model_checkpoint, early_stopping])

print("\n\nUnfreezing base layers for fine-tuning...")
base_model.trainable = True
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_dataset, validation_data=valid_dataset, epochs=EPOCHS+5, initial_epoch=model.history.epoch[-1], callbacks=[model_checkpoint, early_stopping])

model.save('robust_vision_model.keras')
print("Training Complete! Download robust_vision_model.keras and robust_disease_classes.json from the left sidebar.")